# AI4Mars dataset audit v2

Versión corregida para Jupyter/Cursor.

## Mejora principal
- corrige el emparejamiento de MER: las labels de MER a menudo no traen sensor en la ruta,
  así que para los subconjuntos MER se empareja por `(mission, stem_norm)` en lugar de exigir también `sensor`.


In [ ]:
#!/usr/bin/env python3
"""
Audit an AI4Mars-style raw dataset tree and export manifests.

v2 notes
--------
- Keeps the audit-first behavior: it does NOT modify raw files.
- Fixes MER subset pairing by matching MER images/labels on (mission, stem_norm)
  instead of (mission, sensor, stem_norm), because MER labels often do not carry
  an explicit sensor in their path and would otherwise be classified as 'unknown'.
"""

from __future__ import annotations

import argparse
import csv
import json
import re
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable


IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
LABEL_EXTS = {".png", ".jpg", ".jpeg"}

MER_TEST_SUFFIX_RE = re.compile(r"_\d+_t0_merged$", re.IGNORECASE)
MER_TRAIN_SUFFIX_RE = re.compile(r"_merged\d+$", re.IGNORECASE)


@dataclass
class Record:
    kind: str
    path: str
    relpath: str
    mission: str
    sensor: str
    split_hint: str
    stem_raw: str
    stem_norm: str
    ext: str


def path_parts_lower(path: Path) -> list[str]:
    return [p.lower() for p in path.parts]


def infer_mission(rel: Path) -> str:
    parts = path_parts_lower(rel)
    for mission in ("m2020", "mer", "msl"):
        if mission in parts:
            return mission
    return "unknown"


def infer_sensor(rel: Path) -> str:
    parts = path_parts_lower(rel)
    for sensor in ("ncam", "mcam", "hafiq", "nav"):
        if sensor in parts:
            return sensor
    if "eff" in parts:
        return "eff"
    return "unknown"


def infer_split_hint(rel: Path) -> str:
    parts = path_parts_lower(rel)
    hints = [
        "train",
        "val",
        "valid",
        "validation",
        "test",
        "raw_unmerged",
        "merged_unmasked",
        "masked-gold-min1-100agree",
        "masked-gold-min2-100agree",
        "masked-gold-min3-100agree",
        "m2020_geo",
        "nav",
        "eff",
        "images",
        "labels",
    ]
    for hint in hints:
        if hint in parts:
            return hint
    return "unknown"


def normalize_stem(stem: str) -> str:
    """
    Normalize filename stem for image/label matching.

    Rules derived from the dataset:
    - lowercase
    - MER train labels may end with: _merged6, _merged12, ...
    - MER test labels may end with: _16165_T0_merged
    - collapse whitespace
    """
    s = stem.strip().lower()
    s = MER_TEST_SUFFIX_RE.sub("", s)
    s = MER_TRAIN_SUFFIX_RE.sub("", s)
    s = re.sub(r"\s+", " ", s)
    return s


def scan_records(root: Path) -> tuple[list[Record], list[Record]]:
    image_records: list[Record] = []
    label_records: list[Record] = []

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        ext = path.suffix.lower()
        rel = path.relative_to(root)
        rel_lower_parts = path_parts_lower(rel)

        in_images_tree = "images" in rel_lower_parts
        in_labels_tree = "labels" in rel_lower_parts

        if in_images_tree and ext in IMAGE_EXTS:
            image_records.append(
                Record(
                    kind="image",
                    path=str(path.resolve()),
                    relpath=str(rel),
                    mission=infer_mission(rel),
                    sensor=infer_sensor(rel),
                    split_hint=infer_split_hint(rel),
                    stem_raw=path.stem,
                    stem_norm=normalize_stem(path.stem),
                    ext=ext,
                )
            )
        elif in_labels_tree and ext in LABEL_EXTS:
            label_records.append(
                Record(
                    kind="label",
                    path=str(path.resolve()),
                    relpath=str(rel),
                    mission=infer_mission(rel),
                    sensor=infer_sensor(rel),
                    split_hint=infer_split_hint(rel),
                    stem_raw=path.stem,
                    stem_norm=normalize_stem(path.stem),
                    ext=ext,
                )
            )

    return image_records, label_records


def write_csv(path: Path, rows: Iterable[dict]) -> None:
    rows = list(rows)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        with path.open("w", newline="", encoding="utf-8") as f:
            f.write("")
        return

    fieldnames = list(rows[0].keys())
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def build_groups(records: list[Record]) -> dict[tuple[str, str, str], list[Record]]:
    groups: dict[tuple[str, str, str], list[Record]] = defaultdict(list)
    for rec in records:
        key = (rec.mission, rec.sensor, rec.stem_norm)
        groups[key].append(rec)
    return groups


def build_groups_mission_stem(records: list[Record]) -> dict[tuple[str, str], list[Record]]:
    groups: dict[tuple[str, str], list[Record]] = defaultdict(list)
    for rec in records:
        key = (rec.mission, rec.stem_norm)
        groups[key].append(rec)
    return groups


def rows_from_records(records: list[Record]) -> list[dict]:
    return [asdict(r) for r in records]


def contains_path_fragment(rec: Record, frag: str) -> bool:
    return frag.lower() in rec.relpath.lower()


def make_subset_manifest(
    name: str,
    image_records: list[Record],
    label_records: list[Record],
) -> list[dict]:
    img_groups = build_groups(image_records)
    lbl_groups = build_groups(label_records)

    rows: list[dict] = []

    if name == "msl_ncam_v1":
        target = ("msl", "ncam")
        for key, imgs in img_groups.items():
            mission, sensor, stem_norm = key
            if (mission, sensor) != target:
                continue
            lbls = lbl_groups.get(key, [])
            if len(imgs) == 1 and len(lbls) == 1:
                rows.append(
                    {
                        "subset": name,
                        "id": stem_norm,
                        "image_relpath": imgs[0].relpath,
                        "label_relpath": lbls[0].relpath,
                        "mission": mission,
                        "sensor": sensor,
                    }
                )

    elif name == "mer_test_gold_min3":
        mer_test_images = [
            r for r in image_records
            if r.mission == "mer" and contains_path_fragment(r, "/images/test/")
        ]
        mer_gold_labels = [
            r for r in label_records
            if r.mission == "mer" and contains_path_fragment(r, "masked-gold-min3-100agree")
        ]

        img_groups_mer = build_groups_mission_stem(mer_test_images)
        lbl_groups_mer = build_groups_mission_stem(mer_gold_labels)

        for key in sorted(set(img_groups_mer) | set(lbl_groups_mer)):
            imgs = img_groups_mer.get(key, [])
            lbls = lbl_groups_mer.get(key, [])
            mission, stem_norm = key
            if len(imgs) == 1 and len(lbls) == 1:
                img = imgs[0]
                lbl = lbls[0]
                rows.append(
                    {
                        "subset": name,
                        "id": stem_norm,
                        "image_relpath": img.relpath,
                        "label_relpath": lbl.relpath,
                        "mission": mission,
                        "sensor": img.sensor,
                    }
                )

    elif name == "mer_train_candidates":
        mer_eff_images = [
            r for r in image_records
            if r.mission == "mer" and contains_path_fragment(r, "/images/eff/")
        ]
        mer_train_labels = [
            r for r in label_records
            if r.mission == "mer" and contains_path_fragment(r, "merged_unmasked")
        ]

        img_groups_mer = build_groups_mission_stem(mer_eff_images)
        lbl_groups_mer = build_groups_mission_stem(mer_train_labels)

        for key in sorted(set(img_groups_mer) | set(lbl_groups_mer)):
            imgs = img_groups_mer.get(key, [])
            lbls = lbl_groups_mer.get(key, [])
            mission, stem_norm = key
            if len(imgs) == 1 and len(lbls) == 1:
                img = imgs[0]
                lbl = lbls[0]
                rows.append(
                    {
                        "subset": name,
                        "id": stem_norm,
                        "image_relpath": img.relpath,
                        "label_relpath": lbl.relpath,
                        "mission": mission,
                        "sensor": img.sensor,
                    }
                )
    else:
        raise ValueError(f"Unknown subset: {name}")

    rows.sort(key=lambda r: r["id"])
    return rows


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--root", type=Path, required=True, help="Path to the raw AI Mars dataset root")
    parser.add_argument(
        "--out",
        type=Path,
        default=Path("data/processed/manifests"),
        help="Directory where CSV/JSON reports will be written",
    )
    args = parser.parse_args()

    root = args.root.resolve()
    out = args.out.resolve()
    out.mkdir(parents=True, exist_ok=True)

    image_records, label_records = scan_records(root)

    img_groups = build_groups(image_records)
    lbl_groups = build_groups(label_records)

    matched_rows = []
    unmatched_image_rows = []
    unmatched_label_rows = []
    duplicate_image_rows = []
    duplicate_label_rows = []

    for key, recs in img_groups.items():
        if len(recs) > 1:
            mission, sensor, stem_norm = key
            for r in recs:
                duplicate_image_rows.append(
                    {
                        "mission": mission,
                        "sensor": sensor,
                        "stem_norm": stem_norm,
                        "relpath": r.relpath,
                        "ext": r.ext,
                    }
                )

    for key, recs in lbl_groups.items():
        if len(recs) > 1:
            mission, sensor, stem_norm = key
            for r in recs:
                duplicate_label_rows.append(
                    {
                        "mission": mission,
                        "sensor": sensor,
                        "stem_norm": stem_norm,
                        "relpath": r.relpath,
                        "ext": r.ext,
                    }
                )

    all_keys = sorted(set(img_groups.keys()) | set(lbl_groups.keys()))
    for key in all_keys:
        imgs = img_groups.get(key, [])
        lbls = lbl_groups.get(key, [])
        mission, sensor, stem_norm = key

        if len(imgs) == 1 and len(lbls) == 1:
            matched_rows.append(
                {
                    "mission": mission,
                    "sensor": sensor,
                    "stem_norm": stem_norm,
                    "image_relpath": imgs[0].relpath,
                    "label_relpath": lbls[0].relpath,
                }
            )
        else:
            if len(imgs) == 0:
                for l in lbls:
                    unmatched_label_rows.append(
                        {
                            "mission": mission,
                            "sensor": sensor,
                            "stem_norm": stem_norm,
                            "label_relpath": l.relpath,
                            "reason": "label_without_image",
                        }
                    )
            elif len(lbls) == 0:
                for i in imgs:
                    unmatched_image_rows.append(
                        {
                            "mission": mission,
                            "sensor": sensor,
                            "stem_norm": stem_norm,
                            "image_relpath": i.relpath,
                            "reason": "image_without_label",
                        }
                    )
            else:
                for i in imgs:
                    unmatched_image_rows.append(
                        {
                            "mission": mission,
                            "sensor": sensor,
                            "stem_norm": stem_norm,
                            "image_relpath": i.relpath,
                            "reason": f"ambiguous_match_{len(imgs)}imgs_{len(lbls)}labels",
                        }
                    )
                for l in lbls:
                    unmatched_label_rows.append(
                        {
                            "mission": mission,
                            "sensor": sensor,
                            "stem_norm": stem_norm,
                            "label_relpath": l.relpath,
                            "reason": f"ambiguous_match_{len(imgs)}imgs_{len(lbls)}labels",
                        }
                    )

    subset_names = ["msl_ncam_v1", "mer_test_gold_min3", "mer_train_candidates"]
    subset_sizes = {}
    for subset_name in subset_names:
        subset_rows = make_subset_manifest(subset_name, image_records, label_records)
        subset_sizes[subset_name] = len(subset_rows)
        write_csv(out / f"{subset_name}.csv", subset_rows)

    write_csv(out / "images_inventory.csv", rows_from_records(image_records))
    write_csv(out / "labels_inventory.csv", rows_from_records(label_records))
    write_csv(out / "matched_pairs.csv", matched_rows)
    write_csv(out / "unmatched_images.csv", unmatched_image_rows)
    write_csv(out / "unmatched_labels.csv", unmatched_label_rows)
    write_csv(out / "duplicate_images.csv", duplicate_image_rows)
    write_csv(out / "duplicate_labels.csv", duplicate_label_rows)

    summary = {
        "root": str(root),
        "num_images": len(image_records),
        "num_labels": len(label_records),
        "num_matched_pairs": len(matched_rows),
        "num_unmatched_images": len(unmatched_image_rows),
        "num_unmatched_labels": len(unmatched_label_rows),
        "num_duplicate_image_entries": len(duplicate_image_rows),
        "num_duplicate_label_entries": len(duplicate_label_rows),
        "images_by_mission": dict(Counter(r.mission for r in image_records)),
        "labels_by_mission": dict(Counter(r.mission for r in label_records)),
        "images_by_sensor": dict(Counter(r.sensor for r in image_records)),
        "labels_by_sensor": dict(Counter(r.sensor for r in label_records)),
        "subset_sizes": subset_sizes,
    }

    with (out / "summary.json").open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2))


# En notebook no ejecutamos main() automáticamente.


## Uso
Cambia estas rutas y ejecuta la celda.


In [4]:
from pathlib import Path
from collections import Counter
import json

DATASET_ROOT = Path('/home/joel/Documentos/Mars-Rover-Terrain-Intelligence/data/raw/AI4Mars')
OUT_DIR = Path('/home/joel/Documentos/Mars-Rover-Terrain-Intelligence/data/processed/manifests_v2')

root = DATASET_ROOT.resolve()
out = OUT_DIR.resolve()
out.mkdir(parents=True, exist_ok=True)

image_records, label_records = scan_records(root)
img_groups = build_groups(image_records)
lbl_groups = build_groups(label_records)

matched_rows = []
unmatched_image_rows = []
unmatched_label_rows = []
duplicate_image_rows = []
duplicate_label_rows = []

for key, recs in img_groups.items():
    if len(recs) > 1:
        mission, sensor, stem_norm = key
        for r in recs:
            duplicate_image_rows.append({'mission': mission, 'sensor': sensor, 'stem_norm': stem_norm, 'relpath': r.relpath, 'ext': r.ext})

for key, recs in lbl_groups.items():
    if len(recs) > 1:
        mission, sensor, stem_norm = key
        for r in recs:
            duplicate_label_rows.append({'mission': mission, 'sensor': sensor, 'stem_norm': stem_norm, 'relpath': r.relpath, 'ext': r.ext})

all_keys = sorted(set(img_groups.keys()) | set(lbl_groups.keys()))
for key in all_keys:
    imgs = img_groups.get(key, [])
    lbls = lbl_groups.get(key, [])
    mission, sensor, stem_norm = key
    if len(imgs) == 1 and len(lbls) == 1:
        matched_rows.append({'mission': mission, 'sensor': sensor, 'stem_norm': stem_norm, 'image_relpath': imgs[0].relpath, 'label_relpath': lbls[0].relpath})
    else:
        if len(imgs) == 0:
            for l in lbls:
                unmatched_label_rows.append({'mission': mission, 'sensor': sensor, 'stem_norm': stem_norm, 'label_relpath': l.relpath, 'reason': 'label_without_image'})
        elif len(lbls) == 0:
            for i in imgs:
                unmatched_image_rows.append({'mission': mission, 'sensor': sensor, 'stem_norm': stem_norm, 'image_relpath': i.relpath, 'reason': 'image_without_label'})
        else:
            for i in imgs:
                unmatched_image_rows.append({'mission': mission, 'sensor': sensor, 'stem_norm': stem_norm, 'image_relpath': i.relpath, 'reason': f'ambiguous_match_{len(imgs)}imgs_{len(lbls)}labels'})
            for l in lbls:
                unmatched_label_rows.append({'mission': mission, 'sensor': sensor, 'stem_norm': stem_norm, 'label_relpath': l.relpath, 'reason': f'ambiguous_match_{len(imgs)}imgs_{len(lbls)}labels'})

subset_names = ['msl_ncam_v1', 'mer_test_gold_min3', 'mer_train_candidates']
subset_sizes = {}
for subset_name in subset_names:
    subset_rows = make_subset_manifest(subset_name, image_records, label_records)
    subset_sizes[subset_name] = len(subset_rows)
    write_csv(out / f'{subset_name}.csv', subset_rows)

write_csv(out / 'images_inventory.csv', rows_from_records(image_records))
write_csv(out / 'labels_inventory.csv', rows_from_records(label_records))
write_csv(out / 'matched_pairs.csv', matched_rows)
write_csv(out / 'unmatched_images.csv', unmatched_image_rows)
write_csv(out / 'unmatched_labels.csv', unmatched_label_rows)
write_csv(out / 'duplicate_images.csv', duplicate_image_rows)
write_csv(out / 'duplicate_labels.csv', duplicate_label_rows)

summary = {
    'root': str(root),
    'num_images': len(image_records),
    'num_labels': len(label_records),
    'num_matched_pairs': len(matched_rows),
    'num_unmatched_images': len(unmatched_image_rows),
    'num_unmatched_labels': len(unmatched_label_rows),
    'num_duplicate_image_entries': len(duplicate_image_rows),
    'num_duplicate_label_entries': len(duplicate_label_rows),
    'images_by_mission': dict(Counter(r.mission for r in image_records)),
    'labels_by_mission': dict(Counter(r.mission for r in label_records)),
    'images_by_sensor': dict(Counter(r.sensor for r in image_records)),
    'labels_by_sensor': dict(Counter(r.sensor for r in label_records)),
    'subset_sizes': subset_sizes,
}

with (out / 'summary.json').open('w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

summary


{'root': '/home/joel/Documentos/Mars-Rover-Terrain-Intelligence/data/raw/AI4Mars',
 'num_images': 97978,
 'num_labels': 48142,
 'num_matched_pairs': 24284,
 'num_unmatched_images': 73694,
 'num_unmatched_labels': 23858,
 'num_duplicate_image_entries': 574,
 'num_duplicate_label_entries': 1578,
 'images_by_mission': {'mer': 16504, 'msl': 63419, 'm2020': 18055},
 'labels_by_mission': {'mer': 9678, 'msl': 26129, 'm2020': 12335},
 'images_by_sensor': {'eff': 16300,
  'unknown': 204,
  'mcam': 20468,
  'ncam': 58893,
  'hafiq': 2113},
 'labels_by_sensor': {'unknown': 9678,
  'mcam': 15454,
  'ncam': 19183,
  'nav': 3827},
 'subset_sizes': {'msl_ncam_v1': 16064,
  'mer_test_gold_min3': 204,
  'mer_train_candidates': 0}}